In [11]:
import pandas as pd
from pyprojroot import here

In [12]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"

In [13]:
crime = pd.read_csv(RAW_DATA_DIR/"crime/Municipal-Delitos-2015-2025_abr2026.csv", encoding='latin-1')
population = pd.read_csv(INTERIM_DATA_DIR/"population_2015_2025.csv")

In [14]:
crime['CVEGEO'] = crime['Cve. Municipio'].astype(str).str.zfill(5)
population['CVEGEO'] = population['CVEGEO'].astype(str).str.zfill(5)

In [15]:
crime = crime.rename(columns={'Año': 'year'})
crime['total'] = crime.iloc[:, 9:21].sum(axis=1)

In [16]:
A_crime = [ # Criminal activity associated with organized crime by the Federal Law Against Organized Crime
    'Secuestro',
    'Trata de personas',
    'Narcomenudeo',
    'Falsificación',
    'Corrupción de menores'
]

B_crime = [ # Criminal activity associated with organized crime by its general activity
    'Homicidio doloso',
    'Extorsión',
    'Evasión de presos'
]

C_crime = [ # Logistics related crime
    'Robo a transportista',
    'Robo de vehículo automotor'
]

D_crime = [
    'Delitos cometidos por servidores públicos',
    'Electorales'
]

In [17]:
A_df = crime[crime['Subtipo de delito'].isin(A_crime)].copy()
AB_df = crime[crime['Subtipo de delito'].isin(A_crime + B_crime)].copy()
ABC_df = crime[crime['Subtipo de delito'].isin(A_crime + B_crime + C_crime)].copy()
ABCD_df = crime[crime['Subtipo de delito'].isin(A_crime + B_crime + C_crime + D_crime)].copy()

In [18]:
grouped_AB = AB_df.groupby(['year', 'CVEGEO'])['total'].sum().reset_index()
result_AB = grouped_AB.merge(population, on=['year', 'CVEGEO'], how='left')
result_AB.drop(columns='Unnamed: 0', inplace=True)
result_AB['crime_rate'] = result_AB['total'] / result_AB['population'] * 100000

In [19]:
grouped_A = A_df.groupby(['year', 'CVEGEO'])['total'].sum().reset_index()
result_A = grouped_A.merge(population, on=['year', 'CVEGEO'], how='left')
result_A.drop(columns='Unnamed: 0', inplace=True)
result_A['crime_rate'] = result_A['total'] / result_A['population'] * 100000

In [20]:
result_A.to_csv(INTERIM_DATA_DIR/"crime_rates_A.csv", encoding='latin-1')
result_AB.to_csv(INTERIM_DATA_DIR/"crime_rates_AB.csv", encoding='latin-1')

print(f" {len(result_A)} municipalities processed")
print(f" {len(result_AB)} municipalities processed")
print(result_A.sample(10))
print(result_AB.sample(10))


 26153 municipalities processed
 26153 municipalities processed
       year CVEGEO  total  population  crime_rate
1489   2015  28040      5    123761.0    4.040045
14839  2021  20087      1     11331.2    8.825191
25937  2025  30163      0      9901.5    0.000000
15554  2021  22014    660   2149239.9   30.708531
14121  2021  12005      2     15866.7   12.605016
12739  2020  20465      0      3288.0    0.000000
5750   2017  25013      7     84340.1    8.299729
14363  2021  14082      9     74921.6   12.012557
23831  2025  07081      0    118632.5    0.000000
9180   2019  12010      0     56608.4    0.000000
       year CVEGEO  total  population  crime_rate
13400  2020  30038     69    188094.0   36.683786
13258  2020  27016     24    117459.0   20.432662
3323   2016  26034      1      2063.0   48.473097
18763  2023  05030   2245   1853498.4  121.122306
7069   2018  15079      7     27636.0   25.329281
20921  2023  30119      2     11572.8   17.281902
23600  2024  31098      0     31839.